# Model 9 — Traffic Risk Classifier (Decision Tree / ID3)

## Question
> Based on a car's speed, weight, engine power, and drivetrain — does this car fall into a low, medium, or high traffic risk category?

## Introduction
This notebook builds a **Decision Tree classifier** using entropy-based splitting (ID3-style) to classify cars into traffic risk levels.

- **Dataset**: Unscaled data (`proceed_dataset_without_scaling.csv`)
- **Target variable**: `risk_seviyesi` — an engineered column you will create in the next section
- **Required algorithm**: `DecisionTreeClassifier` with `criterion='entropy'` (ID3-style). Do **not** change the criterion.
- **Recommended depth**: `max_depth` between 3 and 6 for interpretability
- **Feature flexibility**: You may choose different features, adjust weights in the composite score, and tune hyperparameters — but you cannot change the general technique category (decision tree / ID3).

### 1. Data Import

The cell below loads all required libraries and the dataset. **Run this cell as-is** — it is provided and complete.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('../../data/proceed_dataset_without_scaling.csv')
print(f"Dataset shape: {df.shape}")
df.head()

### 2. Risk Score Engineering

The cell below engineers the `risk_seviyesi` target column using a composite score approach. Each signal is normalised to 0–1, weighted, and summed. The final score is binned into three equal-frequency classes using quantiles.

**Run this cell as-is**, but you may adjust the weights (the numbers next to each signal) to change how the risk score is computed.

In [ ]:
# Compute a composite risk score from multiple signals.
# Each signal is normalized to 0-1 range, then weighted and summed.
# Higher score = higher traffic risk.

risk_df = df.copy()
scaler_01 = MinMaxScaler()

# Signals that INCREASE risk
risk_cols_pos = []
if 'Maksimum Hız' in risk_df.columns:
    risk_df['norm_maks_hiz'] = scaler_01.fit_transform(risk_df[['Maksimum Hız']].fillna(0))
    risk_cols_pos.append(('norm_maks_hiz', 0.30))  # weight 30%
if 'Motor Gücü' in risk_df.columns:
    risk_df['norm_motor_gucu'] = scaler_01.fit_transform(risk_df[['Motor Gücü']].fillna(0))
    risk_cols_pos.append(('norm_motor_gucu', 0.25))
if 'Hızlanma (0-100)' in risk_df.columns:
    # Lower acceleration time = faster = higher risk → invert
    risk_df['norm_hizlanma_inv'] = 1 - scaler_01.fit_transform(risk_df[['Hızlanma (0-100)']].fillna(risk_df['Hızlanma (0-100)'].median()))
    risk_cols_pos.append(('norm_hizlanma_inv', 0.20))

# Signals that DECREASE risk
risk_cols_neg = []
if 'Ağırlık' in risk_df.columns:
    risk_df['norm_agirlik'] = scaler_01.fit_transform(risk_df[['Ağırlık']].fillna(0))
    risk_cols_neg.append(('norm_agirlik', 0.15))  # heavier = more stable

# Binary signals (0/1)
binary_decreasing = []
if 'Çekiş_AWD (Elektronik)' in risk_df.columns:
    binary_decreasing.append(('Çekiş_AWD (Elektronik)', 0.05))
if 'Yakıt Tipi_Elektrik' in risk_df.columns:
    binary_decreasing.append(('Yakıt Tipi_Elektrik', 0.05))

# Compute composite score
risk_df['risk_score'] = 0.0
for col, w in risk_cols_pos:
    risk_df['risk_score'] += risk_df[col] * w
for col, w in risk_cols_neg:
    risk_df['risk_score'] -= risk_df[col] * w
for col, w in binary_decreasing:
    risk_df['risk_score'] -= risk_df[col].fillna(0) * w

# Bin into 3 classes using quantiles
risk_df['risk_seviyesi'] = pd.qcut(
    risk_df['risk_score'],
    q=3,
    labels=['Düşük', 'Orta', 'Yüksek']
)

print(f"Risk class distribution:\n{risk_df['risk_seviyesi'].value_counts()}")
print(f"\nYou may adjust the weights above to change how the risk score is computed.")

### 3. Feature Selection

**TODO — Student task:** Review the recommended features below. Do **not** include `risk_score` or any intermediate `norm_` columns — these would cause data leakage. After finalizing your features, run this cell to prepare training and test sets.

In [ ]:
# TODO: Select your features. Do NOT use risk_score or intermediate norm_ columns.
recommended_features = [
    'Hızlanma (0-100)', 'Maksimum Hız', 'Ağırlık', 'Çekiş_AWD (Elektronik)',
    'Kasa Tipi_SUV', 'Motor Gücü', 'Yakıt Tipi_Elektrik', 'Boş Ağırlığı', 'Tork'
]
target = 'risk_seviyesi'

features = [f for f in recommended_features if f in risk_df.columns]

X = risk_df[features].fillna(risk_df[features].median())
y = risk_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Class distribution in test:\n{y_test.value_counts()}")

### 4. Model Training

**TODO — Student task:** This cell is a placeholder. The `criterion='entropy'` implements ID3-style information gain splitting — **do not change this**. You may tune `max_depth`, `min_samples_leaf`, and other hyperparameters.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# TODO: Tune max_depth. Recommended range: 3-6 for interpretability.
# criterion='entropy' implements ID3-style information gain splitting.
model = DecisionTreeClassifier(
    criterion='entropy',   # ID3-style — do not change this
    max_depth=4,           # TODO: try 3, 5, 6
    min_samples_leaf=10,   # TODO: adjust for pruning
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("Decision Tree (ID3) trained.")
print(f"Tree depth: {model.get_depth()}, Leaves: {model.get_n_leaves()}")

### 5. Classification Report

This report shows per-class precision, recall, and F1-score. Review all three classes: Düşük (Low), Orta (Medium), and Yüksek (High).

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_pred with your actual outputs after training.
print("=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=['Düşük', 'Orta', 'Yüksek']))

### 6. Decision Tree Visualization

This is the **primary visualization for your presentation slide**. The tree diagram shows every split node with the feature used, entropy value, and sample counts. The saved PNG file can be embedded directly in your slide deck.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace model, features, and class names with your actual trained model after training.
# This is the primary visualization for your presentation slide.
class_names = ['Düşük', 'Orta', 'Yüksek']

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    model,
    feature_names=features,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=10,
    ax=ax,
    impurity=True,
    proportion=False
)
ax.set_title('Decision Tree — Traffic Risk Classifier (ID3 / Entropy)\n(max_depth shown; adjust for clarity)', fontsize=14)
plt.tight_layout()
plt.savefig('decision_tree_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print("Tree visualization saved as 'decision_tree_visualization.png'")

### 7. Confusion Matrix

The confusion matrix shows how often each predicted risk class matches the actual class. Diagonal cells are correct predictions; off-diagonal cells are errors.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace y_test and y_pred with your actual outputs after training.
class_names = ['Düşük', 'Orta', 'Yüksek']
cm = confusion_matrix(y_test, y_pred, labels=class_names)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=1, ax=ax)
ax.set_xlabel('Predicted Risk Level', fontsize=13)
ax.set_ylabel('Actual Risk Level', fontsize=13)
ax.set_title('Confusion Matrix — Traffic Risk Classifier', fontsize=14)
plt.tight_layout()
plt.show()

### 8. Risk Distribution by Body Type

This grouped bar chart shows what proportion of each body type falls into each risk class. A meaningful pattern here — e.g., SUVs being predominantly low risk due to higher weight — supports your analysis.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual dataframe after engineering the target.

In [ ]:
# ⚠️ Replace risk_df with your actual dataframe after engineering the target.
body_type_cols = [c for c in features if 'Kasa Tipi_' in c]
if body_type_cols and len(body_type_cols) > 0:
    body_risk_data = []
    for col in body_type_cols:
        body_name = col.replace('Kasa Tipi_', '')
        mask = risk_df[col] == 1
        if mask.sum() > 10:
            dist = risk_df.loc[mask, 'risk_seviyesi'].value_counts(normalize=True) * 100
            for risk_label in ['Düşük', 'Orta', 'Yüksek']:
                body_risk_data.append({
                    'Body Type': body_name,
                    'Risk Level': risk_label,
                    'Percentage': dist.get(risk_label, 0)
                })

    if body_risk_data:
        body_risk_df = pd.DataFrame(body_risk_data)
        pivot = body_risk_df.pivot(index='Body Type', columns='Risk Level', values='Percentage').fillna(0)
        pivot = pivot.reindex(columns=['Düşük', 'Orta', 'Yüksek'])

        fig, ax = plt.subplots(figsize=(11, 6))
        pivot.plot(kind='bar', ax=ax, color=['#2ecc71', '#f1c40f', '#e74c3c'],
                   edgecolor='white', width=0.7)
        ax.set_xlabel('Body Type', fontsize=13)
        ax.set_ylabel('Percentage of Cars (%)', fontsize=13)
        ax.set_title('Traffic Risk Distribution by Body Type', fontsize=14)
        ax.legend(title='Risk Level')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        plt.show()
    else:
        print("Not enough data per body type for this plot.")
else:
    print("No Kasa Tipi_ columns in features. Skipping body type distribution plot.")

### 9. Feature Importances

This bar chart shows which features the decision tree used most heavily when splitting. High importance for speed-related features confirms they are the primary risk drivers.

> ⚠️ This cell uses placeholder data for illustration. Replace with your actual model outputs after training.

In [ ]:
# ⚠️ Replace model and features with your actual trained model after training.
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Feature', fontsize=13)
ax.set_ylabel('Feature Importance (Entropy-based)', fontsize=13)
ax.set_title('Decision Tree Feature Importances — Traffic Risk', fontsize=14)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## ⚠️ If Your Model Underperforms

If your model produces poor results (e.g., low accuracy, the tree splits on unexpected features, or all cars end up in the same risk class), **do not discard your results**.

- Keep all outputs as-is
- In your presentation, document exactly what you observe
- Write a short hypothesis: Why might the model have failed? (e.g., 'The engineered risk labels may be too noisy because the composite score formula distributes scores evenly by design — the model sees very similar features across classes')